# Is depth of coverage assocated with ecDNA calls?
## Requirements
Data:
- `../data/source/mosdepth` (see also cavatica-api/download-mosdepth-results.ipynb)

Software:
- pandas
- numpy
- seaborn
- scipy
- sklearn

## Results
One-way ANOVA, amplicon class vs depth of coverage for autosomes, no correction for covariates:  
F-statistic: 0.49, p-value: 0.61 (CBTN)  
F-statistic: 17, p-value: 3.6e-08 (CBTN+SJ)  
F-statistic: 6.5, p-value: 0.002 (SJ)  
-->  Strong batch effect

Likelihood Ratio Test, covariates for sex, age, tumor type, batch:  
(CBTN) n=999; p=0.24  
(CBTN+SJ) n=2330; p=0.01  

## Conclusion
Depth of sequencing coverage (autosomes) is predictive of ecDNA status, specifically in the St Jude dataset. Why?

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats
import numpy as np
from sklearn.linear_model import LogisticRegression

import warnings
import sys
from pathlib import Path

sys.path.append('../src')
Path("out").mkdir(parents=True, exist_ok=True)

import data_imports
from likelihood_ratio_test import likelihood_ratio_test
from seaborn_helper_functions import *

pd.set_option('display.max_columns', None)

In [ ]:
AUTOSOMES = ['chr'+str(x) for x in range(1,23)]
PROJ_HOME=Path('..')
MOSDEPTH_DIR=PROJ_HOME/Path('data/source/mosdepth/')

In [ ]:
biosamples = data_imports.import_biosamples()
biosamples[biosamples.index.str.startswith('SJ')]

In [ ]:
def parse_mosdepth_summary(path):
    path = Path(path)
    df = pd.read_csv(path.resolve(),sep='\t',index_col=0)
    # mean across all contigs
    d1 = df.loc['total','mean'] 
    # mean across autosomes
    df.loc['autosomal','mean'] = np.average(df.loc[AUTOSOMES,'mean'],weights=df.loc[AUTOSOMES,'length'])
    return df

def parse_mosdepth_filepath(biosample):
    if biosample.startswith('BS'):
        return MOSDEPTH_DIR/Path('CBTN_mosdepth/'+biosample+'.mosdepth.summary.txt')
    elif biosample.startswith('SJ'):
        return MOSDEPTH_DIR/Path('SJ_mosdepth/'+biosample+'/'+biosample+'.WholeGenome.mosdepth.summary.txt')
    else:
        raise ValueError('biosample name not recognized: '+str(biosample))

def load_coverage(biosample):
    try:
        path = parse_mosdepth_filepath(biosample)
        depth = parse_mosdepth_summary(path)
        return depth.loc['autosomal','mean']
    except FileNotFoundError:
        return np.nan
    except:
        raise

def load_coverage_data(biosamples = None):
    if biosamples is None:
        df = data_imports.import_biosamples()
    else:
        df = biosamples.copy()
    df['coverage'] = df.index.map(load_coverage)
    return df

In [ ]:
example = '/Users/ochapman/projects/pedpancan_ecdna/data/source/mosdepth/CBTN_mosdepth/BS_00DBDSHZ.mosdepth.summary.txt'
parse_mosdepth_summary(example)

In [ ]:
df = load_coverage_data(biosamples)
print(df.shape)
df = df[~df.coverage.isna()]
df['batch'] = df.index.str.startswith('BS')
print(df.shape)

In [ ]:
from scipy.stats import f_oneway

def plot_ecDNA_vs_coverage(data):
    data=data[data['in_unique_tumor_set']] # imperfect but it'll do
    data = data.dropna(subset=['amplicon_class','coverage'])

    # one-way ANOVA:
    groups = [group['coverage'].values for _, group in data.groupby('amplicon_class')]
    f_stat, p_value = f_oneway(*groups)
    print(f"F-statistic: {f_stat}, p-value: {p_value}")
    
    # figure
    fig, ax = plt.subplots(figsize=(5, 4))
    order=sorted(data['amplicon_class'].unique())
    sns.boxplot(data=data,x='amplicon_class',y='coverage',order=order,ax=ax)
    plt.tight_layout()
    sns.despine()
    return fig,ax

set_plot_defaults()
fig, ax = plot_ecDNA_vs_coverage(df)
savefig(fig, 'out/boxplot_coverage')

In [ ]:
fig, ax = plot_ecDNA_vs_coverage(df[df.index.str.startswith('BS')])
savefig(fig, 'out/boxplot_coverage_CBTN')

In [ ]:
df.sort_values('coverage',ascending=True)

In [ ]:
def lrt_coverage(asdf,variable,include_extent_of_resection=False):
    if include_extent_of_resection:
        v1 = ['sex','age_at_diagnosis','extent_of_tumor_resection','cancer_type','batch']
    else:
        v1 = ['sex','age_at_diagnosis','cancer_type','batch']
    v2 = v1 + [variable]
    asdf=asdf[asdf['in_unique_tumor_set']]
    data = asdf[v2+['amplicon_class']].dropna()
    # drop rare values
    if include_extent_of_resection:
        cts = data.extent_of_tumor_resection.value_counts()
        keep = cts[cts >=5].index
        data = data[data.extent_of_tumor_resection.isin(keep)]
    # keep only tumor types with at least 1 example of ecDNA, intrachromosomal, no amp
    amp_classes = set(data['amplicon_class'].unique())
    gb = data.groupby('cancer_type')['amplicon_class'].nunique()
    complete_types = gb[gb == len(amp_classes)].index
    print(complete_types)
    data = data[data['cancer_type'].isin(complete_types)]
    # convert types and normalize
    data['batch'] = df.batch.astype(float)
    for col in ['age_at_diagnosis','coverage']:
        if col in df.columns:
            df[col] = (df[col] - df[col].mean()) / df[col].std(ddof=1)
    print(len(data))
    
    # data definitions
    y = data.amplicon_class
    x1=pd.get_dummies(data[v1],drop_first=True)
    x2=pd.get_dummies(data[v2],drop_first=True)
    
    # models
    model = LogisticRegression(penalty=None,solver='newton-cg')
    
    return likelihood_ratio_test(features_alternate=x2,features_null=x1,lr_model=model,labels=y)

In [ ]:
lrt_coverage(df,'coverage')

In [ ]:
lrt_coverage(df[df.index.str.startswith('BS')],'coverage')